# From scores to a confusion matrix

**Optional code walkthrough for AI 101 (Module 8).** Not graded. Quiz scores do not depend on it.

**Objective.** Turn detector scores into fault decisions with a threshold, count TP, FP, FN and TN, and calculate accuracy, precision, recall and F1. Then see why accuracy alone misleads for rare faults, and how the threshold changes the counts.

**Supports.** Lesson 8.1 (sections “Declare the positive class” and “Select a threshold using validation”), Lesson 8.2 (section “Calculate the supplied example”) and Lesson 8.3 (section “Compare with the majority-class baseline”).

Run the cells from top to bottom. All data are synthetic and invented for teaching.


## About the data

The first example uses ten invented detector scores and the actual classes. A label of 1 means a real fault and 0 means healthy. Fault is the positive class. These are evaluation labels, not a judgement about what is desirable (Lesson 8.1).

The scores are synthetic and are not calibrated probabilities. They are not from a trained or tested machinery detector.

```
Score   0.95 0.80 0.65 0.55 0.45 0.40 0.30 0.20 0.15 0.05
Actual     1    1    0    1    1    0    0    0    0    0   (1 = fault)
```


## Step 1. Apply a threshold and count TP, FP, FN and TN

Lesson 8.1 labels an example as a fault when the score is at least the threshold. You compare each decision with the actual class. A true positive is a fault predicted as a fault. A false negative is a missed fault. A false positive is a false alarm. A true negative is a healthy example predicted as healthy.

In [ ]:
# Step 1: threshold the scores and count the four outcomes (synthetic data)
scores = [0.95, 0.80, 0.65, 0.55, 0.45, 0.40, 0.30, 0.20, 0.15, 0.05]
actual = [1, 1, 0, 1, 1, 0, 0, 0, 0, 0]        # 1 = fault (positive class), 0 = healthy
threshold = 0.5

tp = fp = fn = tn = 0
for score, label in zip(scores, actual):
    predicted_fault = score >= threshold
    if predicted_fault and label == 1:
        tp += 1
    elif predicted_fault and label == 0:
        fp += 1
    elif (not predicted_fault) and label == 1:
        fn += 1
    else:
        tn += 1

print(f"Threshold {threshold}: TP={tp}, FP={fp}, FN={fn}, TN={tn}")

**Expected output**

```
Threshold 0.5: TP=3, FP=1, FN=1, TN=5
```

Four alerts were raised, and three of them were real faults. One real fault, with score 0.45, was missed because it fell below the threshold. One healthy example, with score 0.65, caused a false alarm.

## Step 2. The metrics for the lesson’s counts

Lesson 8.2 gives TP = 8, FP = 2, FN = 4 and TN = 86, so N = 100. Each metric answers a different question. Accuracy: what fraction of all examples were right? Precision: what fraction of alerts were real faults? Recall: what fraction of real faults were found?

In [ ]:
# Step 2: metrics from the lesson's counts
tp, fp, fn, tn = 8, 2, 4, 86
n = tp + fp + fn + tn

accuracy = (tp + tn) / n
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * tp / (2 * tp + fp + fn)

print(f"N = {n}")
print(f"Accuracy:  {accuracy:.1%}")
print(f"Precision: {precision:.1%}")
print(f"Recall:    {recall:.1%}")
print(f"F1:        {f1:.1%}")

**Expected output**

```
N = 100
Accuracy:  94.0%
Precision: 80.0%
Recall:    66.7%
F1:        72.7%
```

These match Lesson 8.2. The accuracy of 94% looks high, but the detector found only 8 of the 12 real faults. Four faults were missed. The high accuracy does not describe fault detection adequately on its own.

## Step 3. The always-healthy baseline

Lesson 8.3 describes a dataset with 99 healthy examples and one fault. A rule that always predicts healthy never raises an alert. You calculate its accuracy and its fault recall.

In [ ]:
# Step 3: the majority-class baseline
healthy = 99
faults = 1
n = healthy + faults

# "always healthy": every healthy example is right, every fault is missed
tp, fp, fn, tn = 0, 0, faults, healthy

accuracy = (tp + tn) / n
recall = tp / (tp + fn)
print(f"Accuracy: {accuracy:.1%}")
print(f"Fault recall: {recall:.1%}")

**Expected output**

```
Accuracy: 99.0%
Fault recall: 0.0%
```

The baseline has 99% accuracy and detects no fault. As Lesson 8.3 says, a high accuracy can belong to a rule that never finds the event of interest. Compare a detector with this baseline and look at positive-class performance.

## Step 4. How the threshold changes the counts

Lesson 8.1 says that lowering the threshold catches more faults but can raise more false alarms. You use the same ten scores from Step 1 with three thresholds. In a real project you would choose the threshold on validation data and then evaluate it once on the test data.

In [ ]:
# Step 4: the same scores with different thresholds
scores = [0.95, 0.80, 0.65, 0.55, 0.45, 0.40, 0.30, 0.20, 0.15, 0.05]
actual = [1, 1, 0, 1, 1, 0, 0, 0, 0, 0]

print("threshold  TP  FP  FN  TN  recall")
for threshold in [0.3, 0.5, 0.7]:
    tp = fp = fn = tn = 0
    for score, label in zip(scores, actual):
        predicted_fault = score >= threshold
        if predicted_fault and label == 1:
            tp += 1
        elif predicted_fault and label == 0:
            fp += 1
        elif (not predicted_fault) and label == 1:
            fn += 1
        else:
            tn += 1
    recall = tp / (tp + fn)
    print(f"{threshold:>9}  {tp:>2}  {fp:>2}  {fn:>2}  {tn:>2}  {recall:.0%}")

**Expected output**

```
threshold  TP  FP  FN  TN  recall
      0.3   4   3   0   3  100%
      0.5   3   1   1   5  75%
      0.7   2   0   2   6  50%
```

A lower threshold finds more faults (recall rises from 50% to 100%) but also raises more false alarms (FP rises from 0 to 3). The scores did not change. Only the decision rule changed. Which row is best depends on the cost of a missed fault compared with a false alarm (Lesson 8.3).

## Try it

Change a value and observe. In the Step 4 code, change `for threshold in [0.3, 0.5, 0.7]:` to `for threshold in [0.1, 0.5, 0.9]:`. Then run Step 4 again.

**Expected result**

```
threshold  TP  FP  FN  TN  recall
      0.1   4   5   0   1  100%
      0.5   3   1   1   5  75%
      0.9   1   0   3   6  25%
```

At 0.1 almost everything is flagged: all four faults are found, but five of the six healthy examples raise an alarm. At 0.9 only the highest score is flagged, so three of the four faults are missed. Extreme thresholds trade one kind of error for the other.

## Limits

- Ten invented scores are far too few to choose a threshold. The example only shows how the counts move.
- The scores are not calibrated probabilities (Lesson 8.1). Choosing a cutoff does not show that a score is a probability.
- If there are no alerts, precision has a zero denominator and needs an explicit reporting rule (Lesson 8.2). The code above does not handle that case.
- Nothing here authorizes deployment. Real alerts need an engineering procedure with defined recipients, evidence and authority (Lesson 8.3).

Lab 3 selects a threshold on validation scores and reports a fixed test result with an always-healthy baseline.